# Multi-Agent Career Orchestrator System

This notebook implements a stateful multi-agent system built using **LangGraph** and the **Groq API** (`llama-3.1-8b-instant`). 

### System Overview
The system matches a candidate's personal profile (education, experience, current role, skills, RIASEC interest profile, and preferences) with **O*NET database attributes** and **BLS (Bureau of Labor Statistics) Outlook and Wage statistics** to design personalized career recommendations, detailed skills gap analyses, career ladders, and current job postings.

---

In [1]:
# 1. Install required packages
!pip install langgraph groq python-dotenv pandas tabulate -q --no-warn-script-location


In [2]:
# 2. Imports and Environment Setup
import os
import json
import time
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from groq import Groq
import pandas as pd
from tabulate import tabulate
from pydantic import BaseModel

# Load environment variables from the parent directory
load_dotenv(dotenv_path="../.env")

groq_key = os.getenv("GROQ_API_KEY")
if not groq_key:
    print("❌ ERROR: GROQ_API_KEY is not set in the .env file. Please check ../.env.")
else:
    print("✅ GROQ_API_KEY loaded successfully. Initializing Groq client...")
    client = Groq()


✅ GROQ_API_KEY loaded successfully. Initializing Groq client...


## 3. Core Database Repositories (O*NET & BLS Data)

We construct a highly detailed mock database containing key O*NET and BLS stats for various occupations to support lookups, calculations, and RAG operations inside our agent system.

In [3]:
# Structured database representing O*NET attributes joined with BLS 2024-2034 Wage Outlook statistics
ONET_BLS_DB = {
    "15-2051.00": {
        "title": "Data Scientists",
        "description": "Develop and implement models to extract insights from structured and unstructured data, using programming and machine learning tools.",
        "skills": {
            "Python": {"importance": 95, "level": 90},
            "SQL": {"importance": 90, "level": 85},
            "Machine Learning": {"importance": 95, "level": 85},
            "Statistics": {"importance": 90, "level": 80},
            "Data Analysis": {"importance": 95, "level": 90},
            "Deep Learning": {"importance": 80, "level": 70},
            "Big Data Technologies": {"importance": 85, "level": 75},
            "Communication": {"importance": 80, "level": 75},
            "Problem Solving": {"importance": 90, "level": 85}
        },
        "riasec": {
            "Realistic": 30,
            "Investigative": 95,
            "Artistic": 40,
            "Social": 60,
            "Enterprising": 70,
            "Conventional": 80
        },
        "tasks": [
            "Clean, preprocess, and structure large datasets.",
            "Build and deploy predictive machine learning and deep learning models.",
            "Perform statistical data analysis and write algorithms in Python/R.",
            "Present complex findings and data dashboards to business stakeholders."
        ],
        "work_context": {
            "wfh_friendly": "High",
            "indoors": "Very High",
            "time_pressure": "Medium",
            "face_to_face": "Medium"
        },
        "education": {
            "level": "Bachelor's degree minimum, Master's/Ph.D. highly preferred",
            "fields": ["Computer Science", "Data Science", "Statistics", "Mathematics"],
            "certifications": ["Google Data Analytics Professional Certificate", "IBM Data Science Professional Certificate"],
            "alternative_paths": ["Bootcamps + strong portfolio of projects", "Self-study + Kaggle competition tracks"],
            "roi_score": "9.5/10"
        },
        "bls": {
            "median_salary": 108000,
            "percentile_25": 85000,
            "percentile_75": 140000,
            "growth_rate": "36% (Much faster than average)",
            "state_wages": {
                "California": 135000,
                "New York": 128000,
                "Texas": 110000,
                "Washington": 132000
            }
        },
        "career_ladder": [
            {"level": 1, "title": "Junior Data Scientist", "years": "0-2"},
            {"level": 2, "title": "Data Scientist", "years": "2-5"},
            {"level": 3, "title": "Senior Data Scientist / Lead Scientist", "years": "5-8"},
            {"level": 4, "title": "Director of Data Science / Principal AI Architect", "years": "8+"}
        ]
    },
    "15-1252.00": {
        "title": "Software Developers",
        "description": "Research, design, and develop computer software systems, applications, and operating network systems.",
        "skills": {
            "Python": {"importance": 90, "level": 90},
            "Java": {"importance": 85, "level": 80},
            "C++": {"importance": 75, "level": 70},
            "SQL": {"importance": 80, "level": 75},
            "System Design": {"importance": 90, "level": 85},
            "Git": {"importance": 85, "level": 85},
            "Docker": {"importance": 80, "level": 75},
            "Cloud Architecture": {"importance": 85, "level": 80},
            "Communication": {"importance": 80, "level": 75},
            "Problem Solving": {"importance": 95, "level": 90}
        },
        "riasec": {
            "Realistic": 40,
            "Investigative": 90,
            "Artistic": 50,
            "Social": 55,
            "Enterprising": 60,
            "Conventional": 75
        },
        "tasks": [
            "Design, code, and test software applications in Python, Java, or C++.",
            "Develop cloud backend microservices and maintain database schemas.",
            "Analyze user needs and translate them into software architectures.",
            "Collaborate with cross-functional product teams using git/agile workflows."
        ],
        "work_context": {
            "wfh_friendly": "Very High",
            "indoors": "Very High",
            "time_pressure": "High",
            "face_to_face": "Medium"
        },
        "education": {
            "level": "Bachelor's degree in Computer Science, Software Engineering or related field",
            "fields": ["Computer Science", "Software Engineering", "Computer Engineering"],
            "certifications": ["AWS Certified Solutions Architect", "Google Cloud Professional Engineer"],
            "alternative_paths": ["Coding Bootcamps", "GitHub Open-Source Contributions & Projects"],
            "roi_score": "9.8/10"
        },
        "bls": {
            "median_salary": 124000,
            "percentile_25": 95000,
            "percentile_75": 155000,
            "growth_rate": "25% (Much faster than average)",
            "state_wages": {
                "California": 148000,
                "New York": 139000,
                "Texas": 118000,
                "Washington": 142000
            }
        },
        "career_ladder": [
            {"level": 1, "title": "Junior Software Developer", "years": "0-2"},
            {"level": 2, "title": "Software Developer", "years": "2-5"},
            {"level": 3, "title": "Senior Software Developer / Tech Lead", "years": "5-8"},
            {"level": 4, "title": "Software Architect / Engineering Manager", "years": "8+"}
        ]
    },
    "15-2051.01": {
        "title": "Business Intelligence Analysts",
        "description": "Produce financial and market intelligence by querying data repositories and generating reports, dashboards, or charts.",
        "skills": {
            "SQL": {"importance": 95, "level": 90},
            "Power BI": {"importance": 90, "level": 85},
            "Tableau": {"importance": 85, "level": 80},
            "Data Analysis": {"importance": 95, "level": 90},
            "Python": {"importance": 75, "level": 70},
            "Data Visualization": {"importance": 95, "level": 85},
            "Business Strategy": {"importance": 80, "level": 75},
            "Communication": {"importance": 90, "level": 85},
            "Problem Solving": {"importance": 85, "level": 80}
        },
        "riasec": {
            "Realistic": 25,
            "Investigative": 82,
            "Artistic": 45,
            "Social": 70,
            "Enterprising": 80,
            "Conventional": 90
        },
        "tasks": [
            "Query databases using SQL to extract raw business data.",
            "Create interactive reports and dashboards using Power BI and Tableau.",
            "Analyze industry trends and competitor data to identify market gaps.",
            "Present regular business metrics to executive leaders and managers."
        ],
        "work_context": {
            "wfh_friendly": "High",
            "indoors": "Very High",
            "time_pressure": "Medium",
            "face_to_face": "High"
        },
        "education": {
            "level": "Bachelor's degree in Business Administration, Finance, CS, or Analytics",
            "fields": ["Business Analytics", "Finance", "Computer Science", "Information Systems"],
            "certifications": ["Microsoft Certified: Power BI Data Analyst Associate", "Tableau Desktop Certified Associate"],
            "alternative_paths": ["Business operations background + online analytics bootcamps"],
            "roi_score": "9.0/10"
        },
        "bls": {
            "median_salary": 95000,
            "percentile_25": 75000,
            "percentile_75": 120000,
            "growth_rate": "15% (Faster than average)",
            "state_wages": {
                "California": 112000,
                "New York": 108000,
                "Texas": 92000,
                "Washington": 104000
            }
        },
        "career_ladder": [
            {"level": 1, "title": "Junior BI Analyst", "years": "0-2"},
            {"level": 2, "title": "BI Analyst", "years": "2-5"},
            {"level": 3, "title": "Senior BI Analyst / Analytics Lead", "years": "5-8"},
            {"level": 4, "title": "BI Architect / Analytics Manager", "years": "8+"}
        ]
    },
    "29-1141.00": {
        "title": "Registered Nurses",
        "description": "Assess patient health problems and needs, develop and implement nursing care plans, and maintain medical records.",
        "skills": {
            "Patient Care": {"importance": 98, "level": 95},
            "Critical Thinking": {"importance": 92, "level": 85},
            "Medicine": {"importance": 90, "level": 80},
            "Communication": {"importance": 95, "level": 90},
            "Active Listening": {"importance": 95, "level": 90},
            "Coordination": {"importance": 88, "level": 80}
        },
        "riasec": {
            "Realistic": 50,
            "Investigative": 75,
            "Artistic": 35,
            "Social": 95,
            "Enterprising": 50,
            "Conventional": 70
        },
        "tasks": [
            "Administer medications and treatments to patients.",
            "Monitor, record, and report symptoms or changes in patient conditions.",
            "Collaborate with doctors and healthcare teams to manage patient care.",
            "Instruct patients and families on health management and disease prevention."
        ],
        "work_context": {
            "wfh_friendly": "Very Low",
            "indoors": "Very High",
            "time_pressure": "Very High",
            "face_to_face": "Very High"
        },
        "education": {
            "level": "Associate's Degree in Nursing (ADN) or Bachelor of Science in Nursing (BSN) + State License (RN)",
            "fields": ["Nursing", "Pre-Medicine", "Biology"],
            "certifications": ["Registered Nurse License (NCLEX-RN)", "Advanced Cardiovascular Life Support (ACLS)"],
            "alternative_paths": ["LPN-to-RN bridge programs", "Accelerated BSN for degree holder in other fields"],
            "roi_score": "8.8/10"
        },
        "bls": {
            "median_salary": 86000,
            "percentile_25": 68000,
            "percentile_75": 105000,
            "growth_rate": "6% (Average)",
            "state_wages": {
                "California": 124000,
                "New York": 98000,
                "Texas": 82000,
                "Florida": 79000
            }
        },
        "career_ladder": [
            {"level": 1, "title": "Staff Nurse (RN)", "years": "0-3"},
            {"level": 2, "title": "Charge Nurse", "years": "3-5"},
            {"level": 3, "title": "Nurse Practitioner (NP) / Clinical Specialist", "years": "5-8 (requires MSN/DNP)"},
            {"level": 4, "title": "Director of Nursing / Healthcare Administrator", "years": "8+"}
        ]
    },
    "15-1212.00": {
        "title": "Information Security Analysts",
        "description": "Plan, implement, upgrade, or monitor security measures for the protection of computer networks and information.",
        "skills": {
            "Information Security": {"importance": 98, "level": 90},
            "Network Security": {"importance": 95, "level": 85},
            "SQL": {"importance": 70, "level": 60},
            "Python": {"importance": 80, "level": 70},
            "Problem Solving": {"importance": 90, "level": 85},
            "System Monitoring": {"importance": 95, "level": 85},
            "Communication": {"importance": 80, "level": 75}
        },
        "riasec": {
            "Realistic": 45,
            "Investigative": 88,
            "Artistic": 35,
            "Social": 55,
            "Enterprising": 65,
            "Conventional": 85
        },
        "tasks": [
            "Monitor network logs to identify cyber incidents and vulnerability leaks.",
            "Conduct security audits and penetration tests on software infrastructures.",
            "Implement data encryption and system firewalls to block unauthorized logins.",
            "Formulate incident response plans and security guidelines for employees."
        ],
        "work_context": {
            "wfh_friendly": "High",
            "indoors": "Very High",
            "time_pressure": "Very High",
            "face_to_face": "Medium"
        },
        "education": {
            "level": "Bachelor's degree in Cybersecurity, CS or related, plus security certifications",
            "fields": ["Cybersecurity", "Computer Science", "Computer Networks"],
            "certifications": ["CompTIA Security+", "Certified Information Systems Security Professional (CISSP)", "CEH (Ethical Hacker)"],
            "alternative_paths": ["Military IT experience + certifications", "Security analyst internships + certifications"],
            "roi_score": "9.2/10"
        },
        "bls": {
            "median_salary": 112000,
            "percentile_25": 88000,
            "percentile_75": 145000,
            "growth_rate": "32% (Much faster than average)",
            "state_wages": {
                "California": 129000,
                "New York": 122000,
                "Texas": 108000,
                "Virginia": 116000
            }
        },
        "career_ladder": [
            {"level": 1, "title": "Junior SOC Analyst", "years": "0-2"},
            {"level": 2, "title": "Security Engineer", "years": "2-5"},
            {"level": 3, "title": "Senior Cybersecurity Consultant", "years": "5-8"},
            {"level": 4, "title": "CISO (Chief Information Security Officer) / Director", "years": "8+"}
        ]
    },
    "15-1299.09": {
        "title": "Computer and Information Systems Managers",
        "description": "Plan, direct, or coordinate activities in fields such as electronic data processing, information systems, systems analysis, and computer programming.",
        "skills": {
            "Leadership": {"importance": 95, "level": 90},
            "Project Management": {"importance": 95, "level": 90},
            "System Analysis": {"importance": 85, "level": 80},
            "Budgeting": {"importance": 80, "level": 75},
            "Communication": {"importance": 95, "level": 90},
            "Risk Management": {"importance": 85, "level": 80},
            "Problem Solving": {"importance": 90, "level": 85}
        },
        "riasec": {
            "Realistic": 35,
            "Investigative": 80,
            "Artistic": 45,
            "Social": 75,
            "Enterprising": 95,
            "Conventional": 85
        },
        "tasks": [
            "Coordinate information technology projects and lead software engineers.",
            "Plan organizational technology upgrades and negotiate licensing agreements.",
            "Manage project budgets, deliverable timelines, and allocate developers.",
            "Provide technical leadership and risk evaluations for enterprise software deployments."
        ],
        "work_context": {
            "wfh_friendly": "High",
            "indoors": "Very High",
            "time_pressure": "Very High",
            "face_to_face": "Very High"
        },
        "education": {
            "level": "Bachelor's degree in CS, IT, or Management + significant technical leadership experience",
            "fields": ["Information Technology", "Computer Science", "Business Administration"],
            "certifications": ["Project Management Professional (PMP)", "Certified ScrumMaster (CSM)"],
            "alternative_paths": ["Software Developer → Team Lead → Engineering Manager", "Tech sales/operations → IT Manager"],
            "roi_score": "9.6/10"
        },
        "bls": {
            "median_salary": 164000,
            "percentile_25": 125000,
            "percentile_75": 195000,
            "growth_rate": "17% (Faster than average)",
            "state_wages": {
                "California": 188000,
                "New York": 179000,
                "Texas": 152000,
                "Virginia": 168000
            }
        },
        "career_ladder": [
            {"level": 1, "title": "Junior Project Coordinator", "years": "0-2"},
            {"level": 2, "title": "Project Manager (PMP)", "years": "2-5"},
            {"level": 3, "title": "Senior Project / Program Manager", "years": "5-8"},
            {"level": 4, "title": "Director of PMO / VP of Engineering", "years": "8+"}
        ]
    }
}
print(f"✅ Successfully loaded mock DB containing {len(ONET_BLS_DB)} high-profile O*NET & BLS occupations!")


✅ Successfully loaded mock DB containing 6 high-profile O*NET & BLS occupations!


## 4. LangGraph Shared State Design

We define the `AgentState` schema using a Pydantic `BaseModel` to guarantee absolute compatibility and error-free validation on Python 3.14.

In [4]:
# Define state schema using Pydantic BaseModel
class AgentState(BaseModel):
    query: str
    candidate_profile: str = ""      # JSON string
    route_to: str = ""               # string
    matched_occupations: str = ""    # JSON string
    skills_gap: str = ""             # JSON string
    economic_outlook: str = ""       # JSON string
    career_ladder: str = ""          # JSON string
    education_details: str = ""      # JSON string
    live_jobs: str = ""              # JSON string
    execution_logs: str = ""         # Newline-separated string
    final_report: str = ""           # Markdown string


## 5. Specialist Agents Implementation

We implement the core specialized nodes as python modules that read/write the state, using custom algorithms for matching combined with the **Groq LLM** for detailed text descriptions and personalized recommendations.

In [5]:
# helper to call Groq Llama 3
def call_groq_model(system_prompt: str, user_prompt: str) -> str:
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.5,
            max_tokens=800
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"[Error calling Groq API: {e}]"

# 1. Personal Career Match Agent node
def personal_career_match_agent(state: AgentState):
    print("--- [Node: Personal Career Match Agent] ---")
    profile = json.loads(state.candidate_profile or "{}")
    user_skills = set(profile.get("skills", []))
    user_riasec = profile.get("interests_ria_sec", {})
    
    matched = []
    for code, data in ONET_BLS_DB.items():
        # Calculate skills overlap
        job_skills = set(data["skills"].keys())
        matching_skills = user_skills.intersection(job_skills)
        skills_score = int(len(matching_skills) / max(len(job_skills), 1) * 100)
        
        # Calculate RIASEC similarity (Cosine similarity approximation)
        job_riasec = data["riasec"]
        dot_prod = sum(user_riasec.get(k, 50) * job_riasec.get(k, 50) for k in job_riasec.keys())
        mag_user = sum(user_riasec.get(k, 50)**2 for k in job_riasec.keys())**0.5
        mag_job = sum(job_riasec.get(k, 50)**2 for k in job_riasec.keys())**0.5
        riasec_score = int((dot_prod / (mag_user * mag_job)) * 100) if (mag_user * mag_job) else 0
        
        # Weighted Match Score
        overall_score = int((skills_score * 0.5) + (riasec_score * 0.5))
        
        # Query Groq to write a personalized fit description
        sys_prompt = "You are a Career counselor. Explain why this occupation is a match or where the gap is in 2 sentences."
        user_prompt = f"Candidate Name: {profile.get('full_name')}, Role: {profile.get('current_role')}. Target Occupation: {data['title']}. Overall Match Score: {overall_score}%. Skills Overlap: {list(matching_skills)}. RIASEC Scores: {user_riasec}. Explain the fit."
        fit_desc = call_groq_model(sys_prompt, user_prompt)
        
        matched.append({
            "occupation": data["title"],
            "soc_code": code,
            "match_score": overall_score,
            "skills_score": skills_score,
            "riasec_score": riasec_score,
            "why_this_fits": fit_desc
        })
    
    # Sort recommendations by match score descending
    matched = sorted(matched, key=lambda x: x["match_score"], reverse=True)
    
    old_logs = state.execution_logs or ""
    new_logs = old_logs + "\n[INFO] Evaluated personal career fits with O*NET DB"
    return {
        "matched_occupations": json.dumps(matched),
        "execution_logs": new_logs.strip()
    }

# 2. O*NET Occupation Deep Profiler & Skills Matcher Node
def onet_occupation_deep_profiler(state: AgentState):
    print("--- [Node: O*NET Occupation Deep Profiler & Skills Matcher] ---")
    profile = json.loads(state.candidate_profile or "{}")
    user_skills = set(profile.get("skills", []))
    matched_list = json.loads(state.matched_occupations or "[]")
    
    if not matched_list:
        # default to Data Scientists if not matched yet
        target_soc = "15-2051.00"
    else:
        target_soc = matched_list[0]["soc_code"]
        
    job_data = ONET_BLS_DB[target_soc]
    job_skills = job_data["skills"]
    
    # Analyze skills gaps
    gaps = []
    for skill, sinfo in job_skills.items():
        if skill not in user_skills:
            gaps.append({
                "skill": skill,
                "importance": sinfo["importance"],
                "required_level": sinfo["level"],
                "status": "Missing / Needs Upskilling"
            })
    
    old_logs = state.execution_logs or ""
    new_logs = old_logs + f"\n[INFO] Extracted skills gap and O*NET breakdown for SOC: {target_soc}"
    return {
        "skills_gap": json.dumps(gaps),
        "execution_logs": new_logs.strip()
    }

# 3. BLS Economic & Compensation Agent Node
def bls_economic_compensation_agent(state: AgentState):
    print("--- [Node: BLS Economic & Compensation Agent] ---")
    matched_list = json.loads(state.matched_occupations or "[]")
    if not matched_list:
        target_soc = "15-2051.00"
    else:
        target_soc = matched_list[0]["soc_code"]
        
    bls_data = ONET_BLS_DB[target_soc]["bls"]
    old_logs = state.execution_logs or ""
    new_logs = old_logs + f"\n[INFO] Loaded BLS salary info for SOC: {target_soc}"
    return {
        "economic_outlook": json.dumps({
            "median_salary": bls_data["median_salary"],
            "percentile_25": bls_data["percentile_25"],
            "percentile_75": bls_data["percentile_75"],
            "growth_rate": bls_data["growth_rate"],
            "state_wages": bls_data["state_wages"]
        }),
        "execution_logs": new_logs.strip()
    }

# 4. Career Pathway & Transition Planner Node
def related_occupations_career_ladder(state: AgentState):
    print("--- [Node: Career Pathway & Transition Planner] ---")
    matched_list = json.loads(state.matched_occupations or "[]")
    if not matched_list:
        target_soc = "15-2051.00"
    else:
        target_soc = matched_list[0]["soc_code"]
        
    job_data = ONET_BLS_DB[target_soc]
    ladder = job_data["career_ladder"]
    edu = job_data["education"]
    
    old_logs = state.execution_logs or ""
    new_logs = old_logs + f"\n[INFO] Generated career ladders and education requirements for SOC: {target_soc}"
    return {
        "career_ladder": json.dumps(ladder),
        "education_details": json.dumps(edu),
        "execution_logs": new_logs.strip()
    }

# 5. Real-time Job Market Agent Node
def real_time_job_market_agent(state: AgentState):
    print("--- [Node: Real-time Job Market Agent] ---")
    matched_list = json.loads(state.matched_occupations or "[]")
    if not matched_list:
        target_title = "Data Scientist"
    else:
        target_title = matched_list[0]["occupation"]
        
    # Simulating live job openings matching target title
    mock_openings = [
        {"title": f"Senior {target_title}", "company": "TechGlobal Inc.", "location": "Remote / Bangalore", "salary": "$135,000 - $160,000", "link": "https://www.linkedin.com/jobs"},
        {"title": f"Associate {target_title}", "company": "GovSystems Technologies", "location": "Hybrid / Delhi NCR", "salary": "$85,000 - $110,000", "link": "https://www.usajobs.gov"},
        {"title": f"{target_title} Lead", "company": "Innovate Finance LLC", "location": "Remote (US/India)", "salary": "$120,000 - $145,000", "link": "https://www.indeed.com"}
    ]
    
    old_logs = state.execution_logs or ""
    new_logs = old_logs + f"\n[INFO] Retrieved {len(mock_openings)} real-time job openings for: {target_title}"
    return {
        "live_jobs": json.dumps(mock_openings),
        "execution_logs": new_logs.strip()
    }

# 6. Output Synthesizer Node
def output_synthesizer(state: AgentState):
    print("--- [Node: Output Synthesizer] ---")
    profile = json.loads(state.candidate_profile or "{}")
    matched_list = json.loads(state.matched_occupations or "[]")
    gaps = json.loads(state.skills_gap or "[]")
    economic = json.loads(state.economic_outlook or "{}")
    ladder = json.loads(state.career_ladder or "[]")
    edu = json.loads(state.education_details or "{}")
    live = json.loads(state.live_jobs or "[]")
    
    # Generate ascii tables
    matched_df = pd.DataFrame(matched_list).head(3)
    matched_table = tabulate(matched_df[["occupation", "match_score", "skills_score", "riasec_score"]], headers="keys", tablefmt="github", showindex=False)
    
    gap_df = pd.DataFrame(gaps).head(5)
    gap_table = tabulate(gap_df, headers="keys", tablefmt="github", showindex=False) if not gap_df.empty else "No significant skill gaps found!"
    
    state_wage_df = pd.DataFrame(list(economic.get("state_wages", {}).items()), columns=["State", "Median Salary (USD)"])
    state_wage_table = tabulate(state_wage_df, headers="keys", tablefmt="github", showindex=False)
    
    # Ask Groq to write a professional synthesized executive summary
    sys_prompt = "You are a senior recruitment manager and career strategist. Synthesize all findings into a unified, encouraging executive summary (Hinglish/English mix)."
    user_prompt = f"Candidate Profile: {profile}. Target Recommendations Table:\n{matched_table}\nSkill Gaps:\n{gap_table}\nOutlook Pay: Median {economic.get('median_salary')} USD, Growth {economic.get('growth_rate')}. Create a summary."
    summary = call_groq_model(sys_prompt, user_prompt)
    
    # Assemble final markdown report
    report = f"""# 🚀 Personalized Career Orchestrator Report

## 📋 Executive Summary
{summary}

## 🎯 1. Top Career Recommendations (O*NET & Interests Similarity)
{matched_table}

**Fit Details:**
1. **{matched_list[0]['occupation']}** (Score: {matched_list[0]['match_score']}%):
   * {matched_list[0]['why_this_fits']}
2. **{matched_list[1]['occupation']}** (Score: {matched_list[1]['match_score']}%):
   * {matched_list[1]['why_this_fits']}

## 🔍 2. Skills Gap & Upskilling Requirements
Below are the critical skills required to fully transition to your top match (**{matched_list[0]['occupation']}**):
{gap_table}

## 📊 3. Economic & Compensation Statistics (BLS OEWS & OOH)
* **National Median Annual Pay:** ${economic.get('median_salary'):,} USD
* **Growth Outlook (2024-2034):** {economic.get('growth_rate')}
* **Salary range:** 25th Percentile: ${economic.get('percentile_25'):,} USD | 75th Percentile: ${economic.get('percentile_75'):,} USD

### State-Specific Wage Levels:
{state_wage_table}

## 🎓 4. Training, Education & ROI Details
* **Required Level:** {edu.get('level')}
* **Key Fields:** {', '.join(edu.get('fields', []))}
* **Certifications:** {', '.join(edu.get('certifications', []))}
* **Alternative Pathway:** {', '.join(edu.get('alternative_paths', []))}
* **Return on Investment (ROI) Score:** {edu.get('roi_score')}

## 🏆 5. Long-Term Career Ladder (O*NET Pathway)
"""
    for level in ladder:
        report += f"* **Level {level['level']}**: {level['title']} (Experience needed: {level['years']} years)\n"
        
    report += "\n## 💼 6. Real-Time Active Job Openings\n"
    for idx, j in enumerate(live, 1):
        report += f"{idx}. **{j['title']}** at *{j['company']}* ({j['location']}) | Pay: {j['salary']} | [Apply Link]({j['link']})\n"
        
    old_logs = state.execution_logs or ""
    new_logs = old_logs + "\n[INFO] Synthesized final multi-agent report"
    return {
        "final_report": report,
        "execution_logs": new_logs.strip()
    }
print("✅ Specialist Agents and Output Synthesizer implemented successfully!")


✅ Specialist Agents and Output Synthesizer implemented successfully!


## 6. Supervisor / Router Logic

The Supervisor agent coordinates the path. It maps the user's intent to the next node in the sequence.

In [6]:
def supervisor_node(state: AgentState):
    print("--- [Node: Supervisor Orchestrator] ---")
    query = state.query.lower()
    old_logs = state.execution_logs or ""
    
    # Check progress via string states
    matched_occupations = state.matched_occupations
    skills_gap = state.skills_gap
    economic_outlook = state.economic_outlook
    career_ladder = state.career_ladder
    live_jobs = state.live_jobs
    final_report = state.final_report
    
    if not matched_occupations or matched_occupations == "[]":
        next_step = "personal_career_match_agent"
    elif not skills_gap or skills_gap == "[]":
        next_step = "onet_occupation_deep_profiler"
    elif not economic_outlook or economic_outlook == "{}":
        next_step = "bls_economic_compensation_agent"
    elif not career_ladder or career_ladder == "[]":
        next_step = "related_occupations_career_ladder"
    elif not live_jobs or live_jobs == "[]":
        next_step = "real_time_job_market_agent"
    elif not final_report:
        next_step = "output_synthesizer"
    else:
        next_step = "FINISH"
        
    print(f"   -> Routing control to: {next_step}")
    new_logs = old_logs + f"\n[SUPERVISOR] Routed control to: {next_step}"
    return {
        "route_to": next_step,
        "execution_logs": new_logs.strip()
    }


## 7. Compile LangGraph StateGraph Workflow

We construct the StateGraph, register our nodes, and configure conditional routing branches to run the sequence until the supervisor signals `FINISH`.

In [7]:
from langgraph.graph import StateGraph, START, END

# 1. Initialize StateGraph
workflow = StateGraph(AgentState)

# 2. Register Nodes
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("personal_career_match_agent", personal_career_match_agent)
workflow.add_node("onet_occupation_deep_profiler", onet_occupation_deep_profiler)
workflow.add_node("bls_economic_compensation_agent", bls_economic_compensation_agent)
workflow.add_node("related_occupations_career_ladder", related_occupations_career_ladder)
workflow.add_node("real_time_job_market_agent", real_time_job_market_agent)
workflow.add_node("output_synthesizer", output_synthesizer)

# 3. Configure Starting Point
workflow.add_edge(START, "supervisor")

# 4. Register Node return flows back to the Supervisor
workflow.add_edge("personal_career_match_agent", "supervisor")
workflow.add_edge("onet_occupation_deep_profiler", "supervisor")
workflow.add_edge("bls_economic_compensation_agent", "supervisor")
workflow.add_edge("related_occupations_career_ladder", "supervisor")
workflow.add_edge("real_time_job_market_agent", "supervisor")
workflow.add_edge("output_synthesizer", "supervisor")

# 5. Define Conditional Router Edge
def route_condition(state: AgentState) -> str:
    target = state.route_to
    if target == "FINISH":
        return END
    return target

workflow.add_conditional_edges(
    "supervisor",
    route_condition,
    {
        "personal_career_match_agent": "personal_career_match_agent",
        "onet_occupation_deep_profiler": "onet_occupation_deep_profiler",
        "bls_economic_compensation_agent": "bls_economic_compensation_agent",
        "related_occupations_career_ladder": "related_occupations_career_ladder",
        "real_time_job_market_agent": "real_time_job_market_agent",
        "output_synthesizer": "output_synthesizer",
        END: END
    }
)

# 6. Compile Graph
app = workflow.compile()
print("✅ LangGraph Agentic Workflow successfully built and compiled!")


✅ LangGraph Agentic Workflow successfully built and compiled!


## 8. Run and Test the Workflow

Let's test the workflow with a candidate looking to transition into a new career path. We supply: 
- Current role: **Junior Data Analyst**
- Skills: **Python, SQL, Data Analysis, Power BI**
- RIASEC Interests: **Investigative: 88, Social: 75, Conventional: 65, Enterprising: 60** (RIASEC scores range 0-100)
- Target preferences: **Remote work, High career growth**

In [8]:
# Define testing candidate profile
candidate_profile = {
    "full_name": "Rahul Sharma",
    "current_role": "Junior Data Analyst",
    "experience_years": 2.5,
    "education": {
        "degree": "Bachelor of Technology",
        "field": "Computer Science",
        "highest_qualification": "B.Tech"
    },
    "skills": [
        "Python", "SQL", "Data Analysis", "Pandas", "Power BI", "Communication", "Problem Solving"
    ],
    "interests_ria_sec": {
        "Realistic": 40,
        "Investigative": 88,
        "Artistic": 55,
        "Social": 75,
        "Enterprising": 60,
        "Conventional": 65
    },
    "preferences": {
        "preferred_locations": ["Remote", "Bangalore", "Hyderabad"],
        "minimum_salary_usd": 85000,
        "work_style": "Hybrid/Remote",
        "work_life_balance_importance": "High",
        "career_growth_priority": "High"
    },
    "personality_traits": ["Analytical", "Curious", "Collaborative"],
    "career_goals": "Data Science ya AI related role mein jaana hai"
}

inputs = {
    "query": "Evaluate my profile and suggest best career matches + skill gap analyses",
    "candidate_profile": json.dumps(candidate_profile),
    "route_to": "",
    "matched_occupations": "[]",
    "skills_gap": "[]",
    "economic_outlook": "{}",
    "career_ladder": "[]",
    "education_details": "{}",
    "live_jobs": "[]",
    "execution_logs": "",
    "final_report": ""
}

print("🚀 Starting execution of the Career Orchestrator multi-agent graph...")
start_time = time.time()
results = app.invoke(inputs)
end_time = time.time()

print(f"\n🎉 Execution completed in {end_time - start_time:.2f} seconds!")

# Print Execution Path Logs
print("\n--- [TRAVERSAL LOGS] ---")
print(results["execution_logs"])

# Render Final Report
from IPython.display import display, Markdown
print("\n--- [FINAL SYNTHESIZED REPORT] ---")
display(Markdown(results["final_report"]))


🚀 Starting execution of the Career Orchestrator multi-agent graph...
--- [Node: Supervisor Orchestrator] ---
   -> Routing control to: personal_career_match_agent
--- [Node: Personal Career Match Agent] ---


--- [Node: Supervisor Orchestrator] ---
   -> Routing control to: onet_occupation_deep_profiler
--- [Node: O*NET Occupation Deep Profiler & Skills Matcher] ---
--- [Node: Supervisor Orchestrator] ---
   -> Routing control to: bls_economic_compensation_agent
--- [Node: BLS Economic & Compensation Agent] ---
--- [Node: Supervisor Orchestrator] ---
   -> Routing control to: related_occupations_career_ladder
--- [Node: Career Pathway & Transition Planner] ---
--- [Node: Supervisor Orchestrator] ---
   -> Routing control to: real_time_job_market_agent
--- [Node: Real-time Job Market Agent] ---
--- [Node: Supervisor Orchestrator] ---
   -> Routing control to: output_synthesizer
--- [Node: Output Synthesizer] ---


--- [Node: Supervisor Orchestrator] ---
   -> Routing control to: FINISH

🎉 Execution completed in 4.86 seconds!

--- [TRAVERSAL LOGS] ---
[SUPERVISOR] Routed control to: personal_career_match_agent
[INFO] Evaluated personal career fits with O*NET DB
[SUPERVISOR] Routed control to: onet_occupation_deep_profiler
[INFO] Extracted skills gap and O*NET breakdown for SOC: 15-2051.01
[SUPERVISOR] Routed control to: bls_economic_compensation_agent
[INFO] Loaded BLS salary info for SOC: 15-2051.01
[SUPERVISOR] Routed control to: related_occupations_career_ladder
[INFO] Generated career ladders and education requirements for SOC: 15-2051.01
[SUPERVISOR] Routed control to: real_time_job_market_agent
[INFO] Retrieved 3 real-time job openings for: Business Intelligence Analysts
[SUPERVISOR] Routed control to: output_synthesizer
[INFO] Synthesized final multi-agent report
[SUPERVISOR] Routed control to: FINISH

--- [FINAL SYNTHESIZED REPORT] ---


# 🚀 Personalized Career Orchestrator Report

## 📋 Executive Summary
**Candidate Profile Summary: Rahul Sharma**

Rahul, a 2.5-year Junior Data Analyst with a B.Tech in Computer Science, is looking for a Data Science or AI-related role. His strengths lie in analytical, curious, and collaborative traits, along with skills in Python, SQL, Data Analysis, and more.

**Career Recommendations:**

Based on our analysis, we recommend the following roles:

1. **Data Scientists**: With a match score of 76 and a RIASEC score of 98, this role aligns well with Rahul's career goals and strengths.
2. **Business Intelligence Analysts**: A match score of 81 and a RIASEC score of 97 make this role a strong contender, leveraging Rahul's analytical skills.

**Skill Gaps:**

Rahul needs upskilling in the following areas:

1. **Tableau**: Importance (85), Required Level (80), Current Status (Missing / Needs Upskilling)
2. **Data Visualization**: Importance (95), Required Level (85), Current Status (Missing / Needs Upskilling)
3. **Business Strategy**: Importance (80), Required Level (75), Current Status (Missing / Needs Upskilling)

**Career Growth Outlook:**

The Data Science field is growing at 15% (Faster than average), with a median salary of 95000 USD. This presents an exciting opportunity for Rahul to upskill and advance his career.

**Recommendations:**

1. **Upskill in Data Visualization and Tableau**: Focus on acquiring skills in these areas to enhance Rahul's profile.
2. **Pursue a Data Science or AI-related role**: Leverage Rahul's strengths and career goals to secure a role that aligns with his aspirations.
3. **Emphasize Business Strategy skills**: Develop skills in business strategy to complement Rahul's analytical strengths.

By addressing these skill gaps and career aspirations, Rahul can unlock a fulfilling and high-growth career in Data Science or AI.

## 🎯 1. Top Career Recommendations (O*NET & Interests Similarity)
| occupation                     |   match_score |   skills_score |   riasec_score |
|--------------------------------|---------------|----------------|----------------|
| Business Intelligence Analysts |            81 |             66 |             97 |
| Information Security Analysts  |            77 |             57 |             97 |
| Data Scientists                |            76 |             55 |             98 |

**Fit Details:**
1. **Business Intelligence Analysts** (Score: 81%):
   * Rahul's skills overlap in areas such as data analysis, problem-solving, and communication, which are essential for a Business Intelligence Analyst role, making him a strong candidate for the position. However, a 19% gap in the overall match score suggests that Rahul may need to develop skills in areas such as business acumen, strategic thinking, or leadership, which are critical components of a Business Intelligence Analyst's role, as indicated by his lower scores in Enterprising and Conventional areas.
2. **Information Security Analysts** (Score: 77%):
   * Based on the provided information, Rahul Sharma's Junior Data Analyst role and skills such as problem-solving, Python, and SQL demonstrate a strong foundation for a career in Information Security Analysts. However, the lower match score (77%) indicates a potential gap, which could be attributed to the relatively low score in the 'Realistic' dimension (40), as Information Security Analysts often require a more technical and detail-oriented approach.

## 🔍 2. Skills Gap & Upskilling Requirements
Below are the critical skills required to fully transition to your top match (**Business Intelligence Analysts**):
| skill              |   importance |   required_level | status                     |
|--------------------|--------------|------------------|----------------------------|
| Tableau            |           85 |               80 | Missing / Needs Upskilling |
| Data Visualization |           95 |               85 | Missing / Needs Upskilling |
| Business Strategy  |           80 |               75 | Missing / Needs Upskilling |

## 📊 3. Economic & Compensation Statistics (BLS OEWS & OOH)
* **National Median Annual Pay:** $95,000 USD
* **Growth Outlook (2024-2034):** 15% (Faster than average)
* **Salary range:** 25th Percentile: $75,000 USD | 75th Percentile: $120,000 USD

### State-Specific Wage Levels:
| State      |   Median Salary (USD) |
|------------|-----------------------|
| California |                112000 |
| New York   |                108000 |
| Texas      |                 92000 |
| Washington |                104000 |

## 🎓 4. Training, Education & ROI Details
* **Required Level:** Bachelor's degree in Business Administration, Finance, CS, or Analytics
* **Key Fields:** Business Analytics, Finance, Computer Science, Information Systems
* **Certifications:** Microsoft Certified: Power BI Data Analyst Associate, Tableau Desktop Certified Associate
* **Alternative Pathway:** Business operations background + online analytics bootcamps
* **Return on Investment (ROI) Score:** 9.0/10

## 🏆 5. Long-Term Career Ladder (O*NET Pathway)
* **Level 1**: Junior BI Analyst (Experience needed: 0-2 years)
* **Level 2**: BI Analyst (Experience needed: 2-5 years)
* **Level 3**: Senior BI Analyst / Analytics Lead (Experience needed: 5-8 years)
* **Level 4**: BI Architect / Analytics Manager (Experience needed: 8+ years)

## 💼 6. Real-Time Active Job Openings
1. **Senior Business Intelligence Analysts** at *TechGlobal Inc.* (Remote / Bangalore) | Pay: $135,000 - $160,000 | [Apply Link](https://www.linkedin.com/jobs)
2. **Associate Business Intelligence Analysts** at *GovSystems Technologies* (Hybrid / Delhi NCR) | Pay: $85,000 - $110,000 | [Apply Link](https://www.usajobs.gov)
3. **Business Intelligence Analysts Lead** at *Innovate Finance LLC* (Remote (US/India)) | Pay: $120,000 - $145,000 | [Apply Link](https://www.indeed.com)
